In [1]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

from transformers import CLIPProcessor, CLIPModel

In [ ]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [2]:
import easyocr
from PIL import Image

reader = easyocr.Reader(['en'])

def process_image_pipeline(image_path):
    image = Image.open(image_path).convert("RGB")

    # OCR bằng EasyOCR
    result = reader.readtext("harvard_0.jpg", detail=0)
    ocr_text = " ".join(result)

    return ocr_text

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


In [10]:
from PIL import Image
import torch
import pytesseract

from transformers import CLIPProcessor, CLIPModel
from transformers import MarianMTModel, MarianTokenizer

In [17]:
def process_image_pipeline(image_path):
    # 1. Mở ảnh
    image = Image.open(image_path).convert("RGB")
    
    # 2. OCR
    ocr_text = pytesseract.image_to_string(image)

    # 3. CLIP
    texts = ["a photo of a building", "a university campus", "a classroom", "a document", "a street view"]
    
    # Encode ảnh
    image_inputs = clip_processor(images=image, return_tensors="pt")
    image_outputs = clip_model.get_image_features(**image_inputs)
    
    # ÉP KIỂU VỀ TENSOR (Để né lỗi BaseModelOutputWithPooling)
    if not isinstance(image_outputs, torch.Tensor):
        image_features = image_outputs.pooler_output
    else:
        image_features = image_outputs
        
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    
    # Encode chữ
    text_inputs = clip_processor(text=texts, return_tensors="pt", padding=True)
    text_outputs = clip_model.get_text_features(**text_inputs)
    
    if not isinstance(text_outputs, torch.Tensor):
        text_features = text_outputs.pooler_output
    else:
        text_features = text_outputs
        
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # Similarity
    similarity = torch.matmul(image_features, text_features.T)
    caption = texts[similarity.argmax().item()]

    # 4. Dịch thuật
    translated = ""
    if ocr_text.strip():
        inputs = mt_tokenizer(ocr_text, return_tensors="pt", padding=True, truncation=True)
        outputs = mt_model.generate(**inputs)
        translated = mt_tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "ocr_text": ocr_text,
        "translated_text": translated,
        "clip_caption": caption
    }

In [18]:
try:
    # Nhớ kiểm tra file "harvard_0.jpg" có nằm cùng thư mục không nhé
    result = process_image_pipeline("harvard_0.jpg")
    print("--- KẾT QUẢ ---")
    print(f"OCR (Gốc): {result['ocr_text']}")
    print(f"Dịch tiếng Việt: {result['translated_text']}")
    print(f"Dự đoán ảnh: {result['clip_caption']}")
except Exception as e:
    print(f"Vẫn còn lỗi: {e}")

--- KẾT QUẢ ---
OCR (Gốc): 
Dịch tiếng Việt: 
Dự đoán ảnh: a street view


In [9]:
import torch
from PIL import Image
import pytesseract
from transformers import CLIPProcessor, CLIPModel, MarianMTModel, MarianTokenizer

# 1. Load CLIP (Dùng để nhận diện nội dung ảnh)
model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(model_name)
clip_processor = CLIPProcessor.from_pretrained(model_name)

# 2. Load MarianMT (Dùng để dịch thuật - ví dụ Anh sang Việt)
# Nếu mày dùng model khác thì thay tên ở đây
mt_model_name = "Helsinki-NLP/opus-mt-en-vi" 
mt_tokenizer = MarianTokenizer.from_pretrained(mt_model_name)
mt_model = MarianMTModel.from_pretrained(mt_model_name)

print("Đã tải xong tất cả các mô hình!")

try:
    result = process_image_pipeline("harvard_0.jpg")
    print("--- KẾT QUẢ ---")
    print(f"Chữ trong ảnh: {result['ocr_text']}")
    print(f"Dịch: {result['translated_text']}")
    print(f"Mô tả ảnh: {result['clip_caption']}")
except Exception as e:
    print(f"Vẫn còn lỗi: {e}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Đã tải xong tất cả các mô hình!
Vẫn còn lỗi: You have to specify input_ids


C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vietr\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
